# Level 2: compare three MLP configurations

Compare the original configuration with two variants, changing one setting at a time. Each configuration runs the same five development folds; no test data is loaded. All runs use seed 42, batches of 64, AdamW with learning rate `1e-3`, and 20 epochs. Each restores the checkpoint with lowest standardized validation MSE.

| Configuration | Hidden widths | Weight decay | Question |
|---|---|---|---|
| original | 32, 16 | 0.0001 | Reference MLP |
| larger | 64, 32 | 0.0001 | Does more capacity help? |
| stronger_decay | 32, 16 | 0.01 | Does stronger weight decay help? |

Keep the constant training-mean baseline: comparing MLPs does not replace checking whether they improve on ignoring spectra.

**Selection criterion fixed before running:** the mean of the two pooled simulation-level RMSE ratios relative to the baseline. Lower is better. This gives the targets equal weight in relative improvement. Also inspect each target's RMSE and MAE separately for tradeoffs.

**Run:** select `.venv/bin/python`, restart the kernel after module edits, and run top to bottom. Models/scalers stay in `comparison_results` in memory. Small JSON results save settings, histories, and metrics, but no weights.

**Fair batch order:** all configurations use a separate shuffle generator with seed 42. Otherwise architecture initialization consumes different amounts of randomness and changes batch order. Every configuration uses this controlled shuffle stream. Shared experiment defaults remain unchanged.

**Scope:** these are development-selected checkpoints with one seed, not final test results or proof of an optimal architecture. Single-thread CPU execution keeps these small matrix operations efficient; the previous thread setting is restored afterward.


## 1. Locate the project and import helpers

The same setup works from the project root, `mlp/`, or `mlp/level2/`. No new packages are required.

In [1]:
from pathlib import Path
import sys
import contextlib
import io
import json

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "mlp" / "data.py").is_file()), None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook inside the cosmoreg project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from mlp import data
from mlp.model import SpectrumMLP
from mlp.experiment import run_experiment

LEVEL_ROOT = PROJECT_ROOT / "mlp" / "level2"
SPECTRA_ROOT = PROJECT_ROOT / "powerspectrum"
LEVEL = 2
FOLDS = range(1, 6)
print("Project:", PROJECT_ROOT)


Project: /home/meatpigeon/Personal/classes/csci494/cosmoreg


## 2. Define configurations and the ranking rule

The settings dictionaries are passed as keyword arguments to `run_experiment`. Shared settings prevent accidental changes beyond the intended capacity or weight-decay change.

```text
score = mean([
    pooled_Omega_m_RMSE / pooled_baseline_Omega_m_RMSE,
    pooled_sigma_8_RMSE / pooled_baseline_sigma_8_RMSE,
])
```

A score below 1 means an average relative improvement. It can still hide a tradeoff between targets, which the per-target table will show.

In [2]:
common_settings = {
    "learning_rate": 1e-3,
    "batch_size": 64,
    "epochs": 20,
    "seed": 42,
    "shuffle_seed": 42,
}
configs = {
    "original": {**common_settings, "hidden_dims": (32, 16), "weight_decay": 1e-4},
    "larger": {**common_settings, "hidden_dims": (64, 32), "weight_decay": 1e-4},
    "stronger_decay": {**common_settings, "hidden_dims": (32, 16), "weight_decay": 1e-2},
}
selection_criterion = "mean pooled simulation RMSE / pooled baseline RMSE across the two targets"
for name, settings in configs.items():
    parameter_count = sum(p.numel() for p in SpectrumMLP(settings["hidden_dims"]).parameters())
    print(f"{name}: hidden={settings['hidden_dims']}, weight_decay={settings['weight_decay']}, parameters={parameter_count}")


original: hidden=(32, 16), weight_decay=0.0001, parameters=2194
larger: hidden=(64, 32), weight_decay=0.0001, parameters=5410
stronger_decay: hidden=(32, 16), weight_decay=0.01, parameters=2194


## 3. Check the five development folds

Verify simulation separation and record validation counts for correct pooling. Only folds 1–5 are loaded.

In [3]:
fold_counts = {}
seen_ids = set()
for fold in FOLDS:
    X, y, ids = data.load_fold(SPECTRA_ROOT, LEVEL, fold)
    unique_ids = set(np.unique(ids).tolist())
    assert not seen_ids.intersection(unique_ids), "Simulation overlap across folds."
    seen_ids.update(unique_ids)
    fold_counts[fold] = len(unique_ids)
    print(f"Fold {fold}: {len(X)} maps, {len(unique_ids)} simulations")
assert len(seen_ids) == 266


Fold 1: 810 maps, 54 simulations
Fold 2: 795 maps, 53 simulations
Fold 3: 795 maps, 53 simulations
Fold 4: 795 maps, 53 simulations
Fold 5: 795 maps, 53 simulations


## 4. Run 15 independent experiments

There are three configurations times five validation folds. Every run starts with fresh weights, optimizer state, and training-fitted scalers. Complete logs are captured in `comparison_logs[name][fold]`; concise summaries remain visible below.

Rerunning this cell replaces all results and retrains all models.

In [4]:
comparison_results = {}
comparison_logs = {}
previous_threads = torch.get_num_threads()
torch.set_num_threads(1)
try:
    for name, settings in configs.items():
        comparison_results[name] = []
        comparison_logs[name] = {}
        for fold in FOLDS:
            print(f"Running {name}, validation fold {fold}...", flush=True)
            log = io.StringIO()
            with contextlib.redirect_stdout(log):
                result = run_experiment(SPECTRA_ROOT, LEVEL, fold, **settings)
            comparison_results[name].append(result)
            comparison_logs[name][fold] = log.getvalue()
            assert result["validation_simulations"] == fold_counts[fold]
            assert len(result["val_history"]) == settings["epochs"]
            assert np.isclose(result["best_val_loss"], min(result["val_history"]))
            print(f"  Best epoch {result['best_epoch']:2d} | validation MSE {result['best_val_loss']:.4f}")
finally:
    torch.set_num_threads(previous_threads)


Running original, validation fold 1...
  Best epoch 13 | validation MSE 0.5941
Running original, validation fold 2...
  Best epoch 13 | validation MSE 0.6978
Running original, validation fold 3...
  Best epoch 19 | validation MSE 0.5878
Running original, validation fold 4...
  Best epoch 12 | validation MSE 0.7356
Running original, validation fold 5...
  Best epoch 13 | validation MSE 0.4926
Running larger, validation fold 1...
  Best epoch 17 | validation MSE 0.5977
Running larger, validation fold 2...
  Best epoch 13 | validation MSE 0.6953
Running larger, validation fold 3...
  Best epoch 17 | validation MSE 0.5791
Running larger, validation fold 4...
  Best epoch  5 | validation MSE 0.7276
Running larger, validation fold 5...
  Best epoch 13 | validation MSE 0.4898
Running stronger_decay, validation fold 1...
  Best epoch 13 | validation MSE 0.5948
Running stronger_decay, validation fold 2...
  Best epoch 13 | validation MSE 0.6958
Running stronger_decay, validation fold 3...
  Bes

## 5. Pool metrics and rank configurations

For RMSE, pool squared errors with simulation counts as weights, then take the square root. For MAE, use a weighted mean. All pooled metrics here are simulation-level and in original units.

Baseline values must match between configurations because their training/validation data and fitted target scalers are the same. Fold RMSE standard deviations describe variation across splits, not independent uncertainty estimates.

In [5]:
summaries = {}
for name, results in comparison_results.items():
    counts = np.array([r["validation_simulations"] for r in results])
    summary = {}
    for key in ("rmse", "mae", "baseline_rmse", "baseline_mae"):
        values = np.stack([r["simulation_metrics"][key] for r in results])
        assert values.shape == (5, 2) and np.isfinite(values).all()
        if key.endswith("rmse"):
            summary[f"pooled_{key}"] = np.sqrt(np.average(values**2, axis=0, weights=counts))
        else:
            summary[f"pooled_{key}"] = np.average(values, axis=0, weights=counts)
        summary[f"fold_mean_{key}"] = values.mean(axis=0)
        summary[f"fold_std_{key}"] = values.std(axis=0, ddof=1)
    assert np.all(summary["pooled_baseline_rmse"] > 0)
    summary["score"] = float(np.mean(summary["pooled_rmse"] / summary["pooled_baseline_rmse"]))
    summary["best_epochs"] = [r["best_epoch"] for r in results]
    summaries[name] = summary

reference = comparison_results["original"]
for name, results in comparison_results.items():
    for result, original in zip(results, reference):
        assert result["validation_fold"] == original["validation_fold"]
        for key in ("baseline_rmse", "baseline_mae"):
            assert np.allclose(result["simulation_metrics"][key], original["simulation_metrics"][key])
        for key in original["scalers"]:
            assert np.array_equal(result["scalers"][key], original["scalers"][key])

ranking = sorted(summaries, key=lambda name: summaries[name]["score"])
print("Configuration ranking (lower score is better):")
for rank, name in enumerate(ranking, 1):
    print(f"{rank}. {name}: score={summaries[name]['score']:.4f}, best epochs={summaries[name]['best_epochs']}")

for i, target in enumerate(reference[0]["target_names"]):
    print(f"\n{target}: pooled simulation metrics across 266 held-out development simulations")
    print("Configuration      RMSE      MAE    RMSE gain   Fold RMSE SD")
    baseline = summaries["original"]["pooled_baseline_rmse"][i]
    for name in ranking:
        s = summaries[name]
        gain = 100 * (1 - s["pooled_rmse"][i] / baseline)
        print(f"{name:16s}  {s['pooled_rmse'][i]:.4f}   {s['pooled_mae'][i]:.4f}   {gain:8.1f}%     {s['fold_std_rmse'][i]:.4f}")
    print(f"{'mean baseline':16s}  {baseline:.4f}   {summaries['original']['pooled_baseline_mae'][i]:.4f}")

leading_config = ranking[0]
print("\nDevelopment-leading configuration:", leading_config)
print("Median best epoch (candidate final duration):", int(np.median(summaries[leading_config]["best_epochs"])))


Configuration ranking (lower score is better):
1. stronger_decay: score=0.6932, best epochs=[13, 13, 19, 11, 13]
2. original: score=0.6933, best epochs=[13, 13, 19, 12, 13]
3. larger: score=0.6938, best epochs=[17, 13, 17, 5, 13]

Omega_m: pooled simulation metrics across 266 held-out development simulations
Configuration      RMSE      MAE    RMSE gain   Fold RMSE SD
stronger_decay    0.0781   0.0668       30.3%     0.0059
original          0.0782   0.0668       30.3%     0.0056
larger            0.0783   0.0666       30.2%     0.0058
mean baseline     0.1121   0.0967

sigma_8: pooled simulation metrics across 266 held-out development simulations
Configuration      RMSE      MAE    RMSE gain   Fold RMSE SD
stronger_decay    0.0806   0.0673       31.1%     0.0075
original          0.0806   0.0673       31.1%     0.0076
larger            0.0806   0.0675       31.1%     0.0080
mean baseline     0.1169   0.1006

Development-leading configuration: stronger_decay
Median best epoch (candidat

## 6. Inspect fold-by-fold tradeoffs

A pooled advantage may come from only a few folds. Compare each variant with the original on the same held-out simulations. Also count improvements against the baseline; a lower RMSE does not guarantee lower MAE.

In [6]:
for i, target in enumerate(reference[0]["target_names"]):
    print(f"\n{target}: simulation RMSE by validation fold")
    print("Fold   Original    Larger    Stronger decay   Baseline")
    for index, fold in enumerate(FOLDS):
        row = [comparison_results[name][index]["simulation_metrics"]["rmse"][i] for name in configs]
        base = reference[index]["simulation_metrics"]["baseline_rmse"][i]
        print(f"{fold:4d}   {row[0]:.4f}      {row[1]:.4f}      {row[2]:.4f}           {base:.4f}")
    for name, results in comparison_results.items():
        rmse_wins = sum(r["simulation_metrics"]["rmse"][i] < r["simulation_metrics"]["baseline_rmse"][i] for r in results)
        mae_wins = sum(r["simulation_metrics"]["mae"][i] < r["simulation_metrics"]["baseline_mae"][i] for r in results)
        original_wins = sum(r["simulation_metrics"]["rmse"][i] < o["simulation_metrics"]["rmse"][i] for r,o in zip(results,reference))
        print(f"{name}: beats baseline RMSE {rmse_wins}/5, MAE {mae_wins}/5; lower RMSE than original {original_wins}/5")



Omega_m: simulation RMSE by validation fold
Fold   Original    Larger    Stronger decay   Baseline
   1   0.0772      0.0791      0.0773           0.1136
   2   0.0802      0.0785      0.0800           0.1151
   3   0.0749      0.0745      0.0747           0.1115
   4   0.0864      0.0869      0.0868           0.1193
   5   0.0715      0.0716      0.0710           0.1000
original: beats baseline RMSE 5/5, MAE 5/5; lower RMSE than original 0/5
larger: beats baseline RMSE 5/5, MAE 5/5; lower RMSE than original 2/5
stronger_decay: beats baseline RMSE 5/5, MAE 5/5; lower RMSE than original 3/5

sigma_8: simulation RMSE by validation fold
Fold   Original    Larger    Stronger decay   Baseline
   1   0.0763      0.0760      0.0764           0.1129
   2   0.0888      0.0893      0.0888           0.1249
   3   0.0795      0.0799      0.0799           0.1125
   4   0.0867      0.0866      0.0864           0.1276
   5   0.0703      0.0695      0.0702           0.1053
original: beats baseline RM

## 7. Save compact, reproducible development results

Save the settings, selection criterion, ranking, fold metrics, and histories as JSON. Model weights and scalers remain in memory after running; persistent final-model saving comes later. This file is a record of the comparison, not a test report.

In [7]:
def json_value(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Cannot serialize {type(value).__name__}")

saved_results = {
    "level": LEVEL,
    "selection_criterion": selection_criterion,
    "configs": configs,
    "validation_simulations": fold_counts,
    "ranking": ranking,
    "summaries": summaries,
    "fold_results": {
        name: [{key: r[key] for key in (
            "level", "validation_fold", "settings", "parameter_count",
            "validation_simulations", "best_epoch", "best_val_loss", "baseline_loss",
            "target_names", "map_metrics", "simulation_metrics", "train_history", "val_history",
        )} for r in results]
        for name, results in comparison_results.items()
    },
}
results_path = LEVEL_ROOT / "level2_config_comparison.json"
results_path.write_text(json.dumps(saved_results, default=json_value, indent=2, allow_nan=False) + "\n")
print("Saved:", results_path)


Saved: /home/meatpigeon/Personal/classes/csci494/cosmoreg/mlp/level2/level2_config_comparison.json


## What this comparison supports

The ranking selects a development-leading configuration under the stated criterion. Three configurations with one seed are a small controlled comparison, not proof of an optimum. The validation folds select checkpoints and configurations, so their scores are development estimates.

Before final fitting, fix the chosen configuration and training duration. A median best epoch is a simple candidate rule; it is not guaranteed optimal on all development data. Additional seeds can assess initialization sensitivity. Keep the test set untouched until those choices are fixed. This notebook does not train a final model or evaluate test data.
